# 01 — Scrape Movie Scripts from IMSDb

This notebook downloads movie scripts from the [Internet Movie Script Database (IMSDb)](https://imsdb.com/) and saves them locally as `.txt` files.

**Run this notebook first** in the pipeline.

**Outputs:**
- `scripts/` — one `.txt` file per downloaded script (~918 files)
- `imsdb_links.csv` — CSV mapping each movie title to its IMSDb URL

## 1. Imports and Scraping Function

Define `obtener_indice_imsdb()`, which fetches the full movie list from IMSDb's index page and maps each title to its script URL.

In [26]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from thefuzz import process, fuzz
import re

def obtener_indice_imsdb():
    """Obtiene el índice completo de películas de IMSDb con sus URLs de scripts."""
    url = "https://imsdb.com/all-scripts.html"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    
    try:
        res = requests.get(url, headers=headers, timeout=20)
        res.encoding = 'utf-8'
        soup = BeautifulSoup(res.text, 'html.parser')
        
        dict_urls = {}
        
        # Obtener todos los links
        all_links = soup.find_all('a', href=True)
        print(f"Total de links encontrados: {len(all_links)}")
        
        # Extraer solo los títulos de películas (que tienen /Movie Scripts/ en href)
        script_count = 0
        for link in all_links:
            href = link.get('href', '')
            titulo_raw = link.get_text(strip=True)
            
            # Filtrar solo links que apunten a scripts de películas
            if '/Movie Scripts/' in href and '.html' in href.lower():
                # Limpiar el título: remover ratings y espacios extra
                titulo = re.sub(r'\s*\d+/10\s*', '', titulo_raw).strip()
                
                # Validaciones
                if not titulo or len(titulo) < 2 or len(titulo) > 300:
                    continue
                
                # Construir URL con el título formateado
                # El formato es: /scripts/TITULO-FORMATEADO.html
                titulo_url = titulo.replace(' ', '-').replace("'", '')
                # Remover caracteres especiales
                titulo_url = re.sub(r'[^a-zA-Z0-9\-]', '', titulo_url)
                # Remover guiones múltiples
                titulo_url = re.sub(r'-+', '-', titulo_url)
                
                url_final = f"https://imsdb.com/scripts/{titulo_url}.html"
                
                # Agregar al diccionario (evitar duplicados)
                if titulo not in dict_urls:
                    dict_urls[titulo] = url_final
                    script_count += 1
        
        print(f"✅ Se extrajeron {script_count} títulos de películas")
        print(f"✅ Se obtuvieron {len(dict_urls)} películas únicas de IMSDb\n")
        
        return dict_urls
        
    except Exception as e:
        print(f"❌ Error al obtener índice: {e}")
        return {}

## 2. Diagnose the IMSDb Index

Fetch the index and inspect the first few results to verify the URL extraction is working.

> **Note:** The first 5 entries are malformed (concatenated titles from the page layout) and are dropped when building the link index.

In [27]:
# DEBUG: Verificar el índice obtenido
print("="*70)
print("DIAGNÓSTICO DEL ÍNDICE")
print("="*70)

# Obtener el índice
indice_online = obtener_indice_imsdb()

print(f"\nTotal de películas en el índice: {len(indice_online)}")

if len(indice_online) > 0:
    print("\nPrimeras 10 películas y sus URLs:")
    print("-"*70)
    for i, (titulo, url) in enumerate(list(indice_online.items())[:10], 1):
        print(f"{i:2d}. {titulo[:50]}")
        print(f"    URL: {url}")
    
    print("\n" + "="*70)
    print("PRUEBA DE DESCARGA")
    print("="*70)
    
    # Probar con la primera película
    primer_titulo, primer_url = list(indice_online.items())[0]
    print(f"\nProbando descarga de: {primer_titulo}")
    print(f"URL: {primer_url}\n")
    
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
        res = requests.get(primer_url, headers=headers, timeout=10)
        res.encoding = 'utf-8'
        res.raise_for_status()
        
        print(f"✅ Status code: {res.status_code}")
        print(f"✅ Content length: {len(res.text)} caracteres")
        
        soup = BeautifulSoup(res.text, 'html.parser')
        pre = soup.find('pre')
        
        if pre:
            texto = pre.get_text()
            tamaño = len(texto.strip())
            print(f"✅ Encontrado <pre> con {tamaño} caracteres")
            print(f"\nPrimeros 300 caracteres:\n{'-'*70}")
            print(texto[:300])
            print(f"{'-'*70}")
        else:
            print("❌ No se encontró <pre>")
            print("\nEtiquetas principales en la página:")
            for tag_name in ['pre', 'div', 'p', 'table', 'body']:
                tags = soup.find_all(tag_name)
                if tags:
                    sizes = [len(tag.get_text()) for tag in tags]
                    print(f"  {tag_name}: {len(tags)} encontrados (tamaños: {sizes[:3]}...)")
                
    except requests.exceptions.Timeout:
        print("❌ Timeout al descargar")
    except requests.exceptions.HTTPError as e:
        print(f"❌ Error HTTP: {e}")
    except Exception as e:
        print(f"❌ Error: {e}")
else:
    print("❌ No se obtuvieron películas del índice")

DIAGNÓSTICO DEL ÍNDICE
Total de links encontrados: 1364
✅ Se extrajeron 1298 títulos de películas
✅ Se obtuvieron 1298 películas únicas de IMSDb


Total de películas en el índice: 1298

Primeras 10 películas y sus URLs:
----------------------------------------------------------------------
 1. PredatorMaster and CommanderWhite ChristmasFantast
    URL: https://imsdb.com/scripts/PredatorMaster-and-CommanderWhite-ChristmasFantastic-Beasts-The-Crimes-of-GrindelwaldLegend.html
 2. Master and CommanderWhite ChristmasFantastic Beast
    URL: https://imsdb.com/scripts/Master-and-CommanderWhite-ChristmasFantastic-Beasts-The-Crimes-of-GrindelwaldLegend.html
 3. White ChristmasFantastic Beasts: The Crimes of Gri
    URL: https://imsdb.com/scripts/White-ChristmasFantastic-Beasts-The-Crimes-of-GrindelwaldLegend.html
 4. Fantastic Beasts: The Crimes of GrindelwaldLegend
    URL: https://imsdb.com/scripts/Fantastic-Beasts-The-Crimes-of-GrindelwaldLegend.html
 5. Legend
    URL: https://imsdb.com/scr

## 3. Build and Save the Link Index

Filter out the malformed first 5 entries, then save the clean title→URL mapping to `imsdb_links.csv`.

In [ ]:
import pandas as pd

# Crear un DataFrame con los títulos y links
df_links = pd.DataFrame(list(indice_online.items()), columns=['Título', 'URL'])

# Ignorar los primeros 5 registros porque da errores for formato de la pagina
df_links_filtered = df_links.iloc[5:].reset_index(drop=True)

# Mostrar todos los links
print(f"Total de películas (sin los primeros 5): {len(df_links_filtered)}\n")
print(df_links_filtered.to_string())

# Guardar a CSV
df_links_filtered.to_csv('imsdb_links.csv', index=False)
print(f"\n✓ Se guardó 'imsdb_links.csv' con todos los links")

## 4. Test Download — First 10 Scripts

Download a small sample before running the full batch to confirm the download and parsing logic works end-to-end.

In [30]:
import os
from pathlib import Path

# Crear carpeta para los scripts de prueba
carpeta_test = "scripts_test"
Path(carpeta_test).mkdir(exist_ok=True)

print("="*70)
print("DESCARGANDO PRIMEROS 10 SCRIPTS")
print("="*70)

# Obtener los primeros 10 registros del DataFrame filtrado
primeros_10 = df_links_filtered.head(10)

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
descargados = 0
errores = 0

for idx, row in primeros_10.iterrows():
    titulo = row['Título']
    url = row['URL']
    
    print(f"\n{idx+1}. Descargando: {titulo}")
    print(f"   URL: {url}")
    
    try:
        res = requests.get(url, headers=headers, timeout=10)
        res.encoding = 'utf-8'
        res.raise_for_status()
        
        # Buscar el script en la página
        soup = BeautifulSoup(res.text, 'html.parser')
        pre = soup.find('pre')
        
        if pre:
            texto = pre.get_text()
            
            # Crear nombre de archivo válido
            nombre_archivo = titulo.replace(' ', '_').replace('/', '_').lower()
            ruta_archivo = os.path.join(carpeta_test, f"{nombre_archivo}.txt")
            
            # Guardar el script
            with open(ruta_archivo, 'w', encoding='utf-8') as f:
                f.write(texto)
            
            tamaño = len(texto)
            print(f"   ✅ Descargado exitosamente ({tamaño} caracteres)")
            descargados += 1
        else:
            print(f"   ❌ No se encontró <pre> en la página")
            errores += 1
            
    except requests.exceptions.Timeout:
        print(f"   ❌ Timeout")
        errores += 1
    except requests.exceptions.HTTPError as e:
        print(f"   ❌ Error HTTP: {e.response.status_code}")
        errores += 1
    except Exception as e:
        print(f"   ❌ Error: {str(e)[:50]}")
        errores += 1

print("\n" + "="*70)
print(f"RESUMEN: {descargados} descargados, {errores} errores")
print(f"Carpeta: {os.path.abspath(carpeta_test)}")
print("="*70)

DESCARGANDO PRIMEROS 10 SCRIPTS

1. Descargando: 10 Things I Hate About You
   URL: https://imsdb.com/scripts/10-Things-I-Hate-About-You.html
   ✅ Descargado exitosamente (215804 caracteres)

2. Descargando: 12
   URL: https://imsdb.com/scripts/12.html
   ✅ Descargado exitosamente (215804 caracteres)

2. Descargando: 12
   URL: https://imsdb.com/scripts/12.html
   ✅ Descargado exitosamente (85610 caracteres)

3. Descargando: 12 and Holding
   URL: https://imsdb.com/scripts/12-and-Holding.html
   ✅ Descargado exitosamente (85610 caracteres)

3. Descargando: 12 and Holding
   URL: https://imsdb.com/scripts/12-and-Holding.html
   ✅ Descargado exitosamente (216091 caracteres)

4. Descargando: 12 Monkeys
   URL: https://imsdb.com/scripts/12-Monkeys.html
   ✅ Descargado exitosamente (216091 caracteres)

4. Descargando: 12 Monkeys
   URL: https://imsdb.com/scripts/12-Monkeys.html
   ✅ Descargado exitosamente (200066 caracteres)

5. Descargando: 12 Years a Slave
   URL: https://imsdb.com/scrip

## 5. Download All Scripts

Download all ~1,293 scripts to the `scripts/` folder. A 0.2s delay is added between requests to avoid overloading the server. Estimated time: ~10 minutes.

In [ ]:
import os
from pathlib import Path
import time

# Folder where all downloaded scripts will be stored
carpeta_completa = "scripts"
Path(carpeta_completa).mkdir(exist_ok=True)

print("="*70)
print("DESCARGANDO TODOS LOS SCRIPTS")
print("="*70)
print(f"Total de películas a descargar: {len(df_links_filtered)}\n")

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
descargados = 0
errores = 0
inicio = time.time()

for idx, row in df_links_filtered.iterrows():
    titulo = row['Título']
    url = row['URL']
    
    # Mostrar progreso cada 50 películas
    if (idx + 1) % 50 == 0:
        elapsed = time.time() - inicio
        print(f"\nProgreso: {idx + 1}/{len(df_links_filtered)} descargadas ({elapsed:.1f}s)")
    
    try:
        res = requests.get(url, headers=headers, timeout=10)
        res.encoding = 'utf-8'
        res.raise_for_status()
        
        # Buscar el script en la página
        soup = BeautifulSoup(res.text, 'html.parser')
        pre = soup.find('pre')
        
        if pre:
            texto = pre.get_text()
            
            # Crear nombre de archivo válido
            nombre_archivo = titulo.replace(' ', '_').replace('/', '_').replace("'", '').lower()
            ruta_archivo = os.path.join(carpeta_completa, f"{nombre_archivo}.txt")
            
            # Guardar el script
            with open(ruta_archivo, 'w', encoding='utf-8') as f:
                f.write(texto)
            
            descargados += 1
        else:
            errores += 1
            
    except requests.exceptions.Timeout:
        errores += 1
    except requests.exceptions.HTTPError:
        errores += 1
    except Exception as e:
        errores += 1
    
    # Pequeña pausa para no sobrecargar el servidor
    time.sleep(0.2)

tiempo_total = time.time() - inicio
print("\n" + "="*70)
print(f"RESUMEN FINAL:")
print(f"✅ Scripts descargados: {descargados}")
print(f"❌ Errores: {errores}")
print(f"⏱️  Tiempo total: {tiempo_total:.1f} segundos")
print(f"📁 Carpeta: {os.path.abspath(carpeta_completa)}")
print("="*70)

## 6. Validate Downloaded Files

Check for empty files (scripts where IMSDb returned an empty `<pre>` block) and remove them to keep the `scripts/` folder clean.

In [32]:
import os

print("="*70)
print("VERIFICANDO ARCHIVOS VACÍOS")
print("="*70)

# Listar todos los archivos en la carpeta
archivos = os.listdir(carpeta_completa)
total_archivos = len(archivos)

archivos_vacios = []
archivos_con_contenido = []

for archivo in archivos:
    ruta_archivo = os.path.join(carpeta_completa, archivo)
    
    if os.path.isfile(ruta_archivo):
        tamaño = os.path.getsize(ruta_archivo)
        
        if tamaño == 0:
            archivos_vacios.append(archivo)
        else:
            archivos_con_contenido.append((archivo, tamaño))

print(f"\nTotal de archivos: {total_archivos}")
print(f"✅ Archivos con contenido: {len(archivos_con_contenido)}")
print(f"❌ Archivos vacíos: {len(archivos_vacios)}")

if archivos_vacios:
    print("\n" + "-"*70)
    print("ARCHIVOS VACÍOS ENCONTRADOS:")
    print("-"*70)
    for archivo in archivos_vacios:
        print(f"  • {archivo}")
    
    # Opcionalmente, eliminar los archivos vacíos
    print("\nEliminando archivos vacíos...")
    for archivo in archivos_vacios:
        ruta = os.path.join(carpeta_completa, archivo)
        os.remove(ruta)
    print(f"✅ Se eliminaron {len(archivos_vacios)} archivos vacíos")
else:
    print("\n✅ No se encontraron archivos vacíos")

print("\n" + "="*70)
print(f"ESTADO FINAL: {len(archivos_con_contenido)} archivos válidos en la carpeta")
print("="*70)

VERIFICANDO ARCHIVOS VACÍOS

Total de archivos: 1260
✅ Archivos con contenido: 918
❌ Archivos vacíos: 342

----------------------------------------------------------------------
ARCHIVOS VACÍOS ENCONTRADOS:
----------------------------------------------------------------------
  • things_my_father_never_taught_me,_the.txt
  • time_machine,_the.txt
  • road,_the.txt
  • theres_something_about_mary.txt
  • hangover,_the.txt
  • planet_of_the_apes,_the.txt
  • freuds_last_session.txt
  • descendants,_the.txt
  • fugitive,_the.txt
  • freddy_vs._jason.txt
  • searchers,_the.txt
  • abyss,_the.txt
  • world_is_not_enough,_the.txt
  • game,_the.txt
  • beekeeper,_the.txt
  • ides_of_march,_the.txt
  • iron_lady,_the.txt
  • lego_movie,_the.txt
  • battle_of_shaker_heights,_the.txt
  • jade.txt
  • fabulous_baker_boys,_the.txt
  • program,_the.txt
  • equilibrium.txt
  • book_of_eli,_the.txt
  • kiss_of_the_spider_woman.txt
  • crow,_the.txt
  • losers,_the.txt
  • millers_crossing.txt
  • ma